# CON .parquet

In [ ]:
#Funciona optimizado
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Script optimizado para generar datasets finales (RF, XGBoost, RNN, LSTM)
- Guarda RF/XGB por estación en Parquet (snappy) y combined por streaming (pyarrow).
- Guarda RNN/LSTM por estación en .npz y combined mediante numpy.memmap.
"""
import sys
import os
import traceback
from pathlib import Path
from typing import List, Optional, Tuple, Dict

import numpy as np
import pandas as pd

# Optional helpers
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PYARROW = True
except Exception:
    PYARROW = False

# For memory info (optional)
try:
    import psutil
    PSUTIL = True
except Exception:
    PSUTIL = False

# ---------------------- CONFIGURACIÓN: ajústala ----------------------
FILE_PATHS = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
OUTPUT_DIR = Path(r"/Volumes/copia seguridad1/Datos TFG").expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72

IMPUTE_STRATEGIES = {
    'NO': 'time_interpolate',
    'NO2': 'time_interpolate',
    'NOx': 'time_interpolate',
    'O3': 'time_interpolate',
    'Veloc.': 'time_interpolate',
    'Direc.': 'circular',
    'Temp.': 'time_interpolate',
    'R.Sol.': 'time_interpolate',
    'Estacion': 'ffill',
    'Transecto': 'ffill',
    'Dist.': 'ffill',
    'Angulo': 'circular'
}
BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']
CATEGORICAL = ['Estacion', 'Transecto']
CIRCULAR_ORIG = ['Direc.', 'Angulo']
TARGET = 'O3'

# 안전 한 상수
MAX_REINDEX_ROWS = 5_000_000         # evitar reindexaciones monstruosas
MAX_SPAN_HOURS = 3_000_000          # umbral conservador de rango temporal (≈342 años)

# ---------------------- UTILIDADES ----------------------

def log(msg: str):
    print(msg, flush=True)

def human_size(nbytes: int) -> str:
    for unit in ['B','KB','MB','GB','TB']:
        if abs(nbytes) < 1024.0:
            return f"{nbytes:3.1f}{unit}"
        nbytes /= 1024.0
    return f"{nbytes:.1f}PB"

def memory_info() -> str:
    if PSUTIL:
        mem = psutil.virtual_memory()
        return f"mem: {human_size(mem.used)}/{human_size(mem.total)} ({mem.percent}%)"
    return "mem: psutil not installed"

# ---------------------- PREPROCESADO ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df = df.dropna(subset=[col])
        df = df.set_index(col)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No hay índice datetime válido")
    df = df.sort_index()
    if not df.index.is_unique:
        df = df[~df.index.duplicated(keep='first')]

    span_hours = int((df.index.max() - df.index.min()) / np.timedelta64(1, 'h')) + 1
    if span_hours > MAX_SPAN_HOURS:
        raise ValueError(f"Rango temporal enorme: {span_hours} horas. Revisa el CSV (fechas erróneas).")

    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    if len(full_idx) - len(df) > MAX_REINDEX_ROWS:
        raise ValueError("La reindexación agregaría demasiadas filas vacías (posible fechas erróneas).")

    df = df.reindex(full_idx)
    return df

def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)
    years = year - year.min()
    denom = max(year.max() - year.min() + 1, 1)
    d['year_sin'] = np.sin(2 * np.pi * years / denom)
    d['year_cos'] = np.cos(2 * np.pi * years / denom)
    return d

def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    radians = np.deg2rad(series)
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(radians),
        f'{prefix}_cos': np.cos(radians)
    }, index=series.index)

def impute_variable(df: pd.DataFrame, col: str, strategy: str) -> pd.Series:
    s = df[col]
    if strategy == 'time_interpolate':
        s_imputed = s.interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strategy == 'ffill':
        return s.ffill().bfill()
    elif strategy == 'median':
        return s.fillna(s.median())
    elif strategy == 'constant':
        return s.fillna(0)
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric types to reduce memory (in-place copy)."""
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col].dtype):
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif pd.api.types.is_integer_dtype(df[col].dtype):
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

def preprocess_single_csv(path: str) -> pd.DataFrame:
    """
    Lectura robusta e imputación.
    """
    p = Path(path)
    log(f"  Leyendo (primeros 5k filas) {p.name} ...")
    # Intentar parse_dates para datetime (si existe)
    try:
        df = pd.read_csv(p, parse_dates=['datetime'], low_memory=False)
    except ValueError:
        # no tiene 'datetime' en header o error; leer sin parse y convertir luego
        df = pd.read_csv(p, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    df = ensure_datetime_index(df, 'datetime')
    df = add_cyclical_datetime_features(df)
    for circ in CIRCULAR_ORIG:
        if circ in df.columns:
            sincos = circular_to_sincos(df[circ], circ.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)
            df.drop(columns=[circ], inplace=True)

    # map de imputación
    impute_map = {}
    for col in df.columns:
        if col in IMPUTE_STRATEGIES:
            strat = IMPUTE_STRATEGIES[col]
            if strat == 'circular':
                strat = 'time_interpolate'
            impute_map[col] = strat
        elif col in BASE_FEATURES or col == TARGET or col.endswith('_sin') or col.endswith('_cos'):
            impute_map[col] = 'time_interpolate'
        elif col in CATEGORICAL:
            impute_map[col] = 'ffill'
        else:
            impute_map[col] = IMPUTE_STRATEGIES.get(col, 'median')

    for col, strat in impute_map.items():
        if col in df.columns:
            df[col] = impute_variable(df, col, strat)

    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    # reducir memoria donde sea posible
    df = reduce_mem_usage(df)
    return df

# ---------------------- VENTANAS DESLIZANTES ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: Optional[List[str]] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf') -> Tuple:
    """
    Devuelve:
      - Para 'rf'/'xgb': (X_df, y_df) DataFrames
      - Para 'rnn'/'lstm': (X_array, y_array) numpy arrays (3D / 2D)
    Implementado de forma eficiente.
    """
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns.tolist()
        if target_col in features:
            features.remove(target_col)
    features = [f for f in features if f in df.columns]

    total_len = len(df)
    if total_len < input_hours + output_hours:
        # estructuras vacías
        if model_type in ['rf','xgb']:
            return pd.DataFrame(), pd.DataFrame()
        else:
            return np.empty((0, input_hours, len(features))), np.empty((0, output_hours))

    # Pre-allocación aproximada: no sabemos cuántas ventanas válidas (sin NaNs) habrá;
    # iremos acumulando por bloques pequeños para evitar listas enormes.
    X_blocks = []
    y_blocks = []
    idxs = []

    for start in range(total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours
        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        X_blocks.append(Xw)
        y_blocks.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        if len(X_blocks) == 0:
            return pd.DataFrame(), pd.DataFrame()
        X_flat = np.array(X_blocks)  # (n, t, f)
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_blocks)
        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d, index=idxs)
        y_df = pd.DataFrame(y_2d, columns=y_cols, index=idxs)
        return X_df, y_df
    else:
        return np.array(X_blocks), np.array(y_blocks)

# ---------------------- FUNCIONES PARA COMBINAR ----------------------

def write_parquet_streaming(df: pd.DataFrame, out_path: Path, parquet_schema: Optional[pa.Schema] = None, compression: str = 'snappy'):
    """
    Escribe (o anexa) un DataFrame a out_path usando pyarrow ParquetWriter.
    Si out_path existe, abre en modo append escribiendo el fragmento nuevo.
    """
    if not PYARROW:
        raise RuntimeError("pyarrow es necesario para escritura por streaming. Instálalo: pip install pyarrow")

    table = pa.Table.from_pandas(df)
    if not out_path.exists():
        writer = pq.ParquetWriter(out_path, table.schema, use_dictionary=True)
        writer.write_table(table)
        writer.close()
    else:
        # append mode: abrimos un writer con el schema existente
        # Simplest: usar parquet.write_table para agregar (pyarrow 3+ permite append a un archivo existente si se abre en modo 'append' con ParquetWriter).
        # Para compatibilidad segura: leemos schema del archivo existente y escribimos nuevo fragmento con append.
        existing = pq.ParquetFile(str(out_path))
        existing_schema = existing.schema.to_arrow_schema()
        # if schemas differ, try to unify by casting columns present in df
        table = table.cast(existing_schema, safe=False)
        with pq.ParquetWriter(out_path, existing_schema, use_dictionary=True) as writer:
            writer.write_table(table)

# ---------------------- MAIN: procesamiento por estación ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    log_dir = output_dir / 'logs'
    per_station_dir = output_dir / 'per_station'
    combined_dir = output_dir / 'combined'
    log_dir.mkdir(parents=True, exist_ok=True)
    per_station_dir.mkdir(parents=True, exist_ok=True)
    combined_dir.mkdir(parents=True, exist_ok=True)

    # Registros para combinar RNN después
    rnn_counts = []  # list of (station_dir / filename, n_samples, input_hours, output_hours, feat_count)

    iterator = file_paths if not TQDM else tqdm(file_paths, desc="Estaciones")
    for p in iterator:
        p = str(Path(p).expanduser())
        station_name = Path(p).stem
        station_dir = per_station_dir / station_name
        station_dir.mkdir(parents=True, exist_ok=True)
        try:
            log(f"\nProcesando estación: {station_name}  ({memory_info()})")
            df = preprocess_single_csv(p)

            # ----- RF & XGB (one-hot for categoricals) -----
            cat_cols = [c for c in CATEGORICAL if c in df.columns]
            df_tab = pd.get_dummies(df, columns=cat_cols)

            feature_cols_tab = df_tab.select_dtypes(include=[np.number]).columns.tolist()
            if TARGET in feature_cols_tab:
                feature_cols_tab.remove(TARGET)

            X_rf, y_rf = make_sliding_windows(df_tab,
                                              features=feature_cols_tab,
                                              target_col=TARGET,
                                              model_type='rf')

            # Guardar por estación (si hay datos)
            if not X_rf.empty:
                X_rf.to_parquet(station_dir / 'rf_X.parquet', compression='snappy', index=True, engine='pyarrow')
                y_rf.to_parquet(station_dir / 'rf_y.parquet', compression='snappy', index=True, engine='pyarrow')
                log(f"  Guardado RF (parquet) -> {station_dir / 'rf_X.parquet'}")

            # XGB (mismo contenido que RF en este flujo)
            X_xgb, y_xgb = make_sliding_windows(df_tab,
                                                features=feature_cols_tab,
                                                target_col=TARGET,
                                                model_type='xgb')
            if not X_xgb.empty:
                X_xgb.to_parquet(station_dir / 'xgb_X.parquet', compression='snappy', index=True, engine='pyarrow')
                y_xgb.to_parquet(station_dir / 'xgb_y.parquet', compression='snappy', index=True, engine='pyarrow')
                log(f"  Guardado XGB (parquet) -> {station_dir / 'xgb_X.parquet'}")

            # ----- RNN/LSTM (sin one-hot) -----
            feature_cols_rnn = df.select_dtypes(include=[np.number]).columns.tolist()
            if TARGET in feature_cols_rnn:
                feature_cols_rnn.remove(TARGET)

            X_rnn, y_rnn = make_sliding_windows(df,
                                                features=feature_cols_rnn,
                                                target_col=TARGET,
                                                model_type='rnn')

            # Guardar npz por estación (incluso si vacío, guardamos shape)
            np.savez_compressed(station_dir / 'rnn_lstm_data.npz', X=X_rnn, y=y_rnn)
            rnn_counts.append((station_dir / 'rnn_lstm_data.npz', X_rnn.shape[0], X_rnn.shape[1] if X_rnn.size else 0, y_rnn.shape[1] if y_rnn.size else 0, len(feature_cols_rnn)))
            log(f"  Guardado RNN npz -> {station_dir / 'rnn_lstm_data.npz'} (n_samples={X_rnn.shape[0]})")

            log(f"  Ventanas: RF={len(X_rf) if hasattr(X_rf,'shape') else 0}, XGB={len(X_xgb) if hasattr(X_xgb,'shape') else 0}, RNN={X_rnn.shape[0] if hasattr(X_rnn,'shape') else 0}")
        except Exception as e:
            # Guardar traza de error
            logf = log_dir / f"{station_name}_error.log"
            with open(logf, 'w', encoding='utf-8') as fh:
                fh.write(f"Error procesando {station_name} ({p}):\n")
                fh.write(traceback.format_exc())
            log(f"  ¡Error procesando {station_name}! Ver {logf}")

    # ---------------------- Crear combinados para RF/XGB via pyarrow streaming ----------------------
    if not PYARROW:
        log("pyarrow no disponible: no se pueden crear combinados por streaming. Instala pyarrow (pip install pyarrow) y vuelve a ejecutar si quieres archivos combined/parquet por streaming.")
    else:
        # Para cada tipo de archivo, buscar por estaciones y escribir en combined/parquet en streaming
        def combine_parquets(pattern: str, out_name: str):
            parts = list(per_station_dir.rglob(pattern))
            if not parts:
                log(f"  No se encontraron archivos {pattern}")
                return
            out_path = combined_dir / out_name
            # Si existe, lo borramos para reescribir de cero
            if out_path.exists():
                out_path.unlink()
            writer = None
            for i, part in enumerate(parts):
                try:
                    df_part = pd.read_parquet(part, engine='pyarrow')
                    table = pa.Table.from_pandas(df_part)
                    if writer is None:
                        writer = pq.ParquetWriter(out_path, table.schema, use_dictionary=True)
                    writer.write_table(table)
                except Exception as e:
                    log(f"  Error añadiendo {part} al combinado: {e}")
            if writer:
                writer.close()
                log(f"  Combined escrito: {out_path} (parts={len(parts)})")

        combine_parquets("rf_X.parquet", "rf_X.parquet")
        combine_parquets("rf_y.parquet", "rf_y.parquet")
        combine_parquets("xgb_X.parquet", "xgb_X.parquet")
        combine_parquets("xgb_y.parquet", "xgb_y.parquet")

    # ---------------------- Combinar RNN/LSTM (.npz) en un único .npz por memmap ----------------------
    # Primera pasada: contar total de muestras
    total_samples = 0
    max_in = 0
    max_out = 0
    feat_count = None
    valid_npzs = []
    for npz_path, n_samples, in_h, out_h, fcount in rnn_counts:
        if n_samples > 0:
            valid_npzs.append((npz_path, n_samples, in_h, out_h, fcount))
            total_samples += n_samples
            max_in = max(max_in, in_h)
            max_out = max(max_out, out_h)
            if feat_count is None:
                feat_count = fcount
            else:
                if fcount != feat_count:
                    log("Advertencia: número de features difiere entre estaciones. Asegúrate de que las features son consistentes.")

    if total_samples == 0:
        log("No hay muestras RNN para combinar.")
    else:
        log(f"Combinando RNN/LSTM: total_samples={total_samples}, input_hours={max_in}, output_hours={max_out}, feat_count={feat_count}")
        # Crear memmap para X (float32) y archivo final npz temporal
        combined_X_path = combined_dir / 'rnn_X.dat'
        combined_y_path = combined_dir / 'rnn_y.dat'
        # eliminar si existen
        if combined_X_path.exists(): combined_X_path.unlink()
        if combined_y_path.exists(): combined_y_path.unlink()

        # Crear memmap con shape (total_samples, max_in, feat_count)
        X_mem = np.memmap(str(combined_X_path), dtype='float32', mode='w+', shape=(total_samples, max_in, feat_count))
        y_mem = np.memmap(str(combined_y_path), dtype='float32', mode='w+', shape=(total_samples, max_out))

        write_ptr = 0
        for npz_path, n_samples, in_h, out_h, fcount in valid_npzs:
            try:
                with np.load(npz_path) as data:
                    X_part = data['X']  # shape (n_samples_part, in_h, fcount)
                    y_part = data['y']  # shape (n_samples_part, out_h)
                    # if in_h < max_in, pad with zeros at front (or back) - here we'll pad at front (earlier timesteps missing)
                    if X_part.shape[1] < max_in:
                        pad_width = max_in - X_part.shape[1]
                        pad_shape = (X_part.shape[0], pad_width, X_part.shape[2])
                        X_pad = np.zeros(pad_shape, dtype=X_part.dtype)
                        X_part = np.concatenate([X_pad, X_part], axis=1)
                    if y_part.shape[1] < max_out:
                        pad_width = max_out - y_part.shape[1]
                        y_pad = np.zeros((y_part.shape[0], pad_width), dtype=y_part.dtype)
                        y_part = np.concatenate([y_part, y_pad], axis=1)
                    # write into memmap region
                    X_mem[write_ptr:write_ptr + X_part.shape[0], :, :] = X_part.astype('float32')
                    y_mem[write_ptr:write_ptr + y_part.shape[0], :] = y_part.astype('float32')
                    write_ptr += X_part.shape[0]
                    log(f"  Copiadas {X_part.shape[0]} muestras desde {npz_path.name} (ptr={write_ptr}/{total_samples})")
            except Exception as e:
                log(f"  Error leyendo {npz_path}: {e}")

        # flush memmaps
        X_mem.flush()
        y_mem.flush()

        # Guardar a npz comprimido final (se crea cargando por streaming en trozos para no duplicar memoria)
        # Numpy no tiene writer comprimido por streaming así que aquí podemos cargar memmap por trozos y escribir un .npz temporal manualmente.
        combined_npz_path = combined_dir / 'rnn_lstm_data_combined.npz'
        # Para evitar usar demasiada memoria, cargamos por filas y acumulamos en archivos .npy temporales
        tmp_X_npy = combined_dir / 'tmp_rnn_X.npy'
        tmp_y_npy = combined_dir / 'tmp_rnn_y.npy'
        # eliminar anteriores
        for q in (tmp_X_npy, tmp_y_npy, combined_npz_path):
            if q.exists():
                q.unlink()
        # Guardar memmap a npy (esto escribe el array completo a disco sin cargarlo todo en memoria)
        np.save(tmp_X_npy, X_mem)  # escribe .npy
        np.save(tmp_y_npy, y_mem)
        # Ahora convertir a npz comprimido leyendo desde los npy (esto requiere memoria al crear el npz, pero es inevitable para .npz).
        # Para mitigar, insertamos un pequeño informe y luego intentamos guardarlo. Si la memoria no permite, el usuario puede mantener los .npy como dato combinado.
        try:
            X_comb = np.load(tmp_X_npy, mmap_mode='r')
            y_comb = np.load(tmp_y_npy, mmap_mode='r')
            np.savez_compressed(combined_npz_path, X=X_comb, y=y_comb)
            log(f"  Combined RNN npz guardado: {combined_npz_path} (samples={total_samples})")
            # opcional: borrar temporales
            tmp_X_npy.unlink()
            tmp_y_npy.unlink()
            combined_X_path.unlink()
            combined_y_path.unlink()
        except Exception as e:
            log(f"  No se pudo crear .npz comprimido final (posible falta de memoria). Los archivos intermedios .npy están en {combined_dir}. Puedes convertirlos manualmente.")
    log("\n¡Proceso completado! Archivos en: " + str(output_dir))

# ---------------------- ENTRYPOINT ----------------------

if __name__ == '__main__':
    if not PYARROW:
        log("Aviso: pyarrow no está instalado. Instálalo para combinados Parquet por streaming: pip install pyarrow")
    try:
        prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)
    except Exception:
        traceback.print_exc()
        sys.exit(1)


Procesando estación: T1_E1_Alicante  (mem: 3.5GB/8.0GB (62.1%))
  Leyendo (primeros 5k filas) T1_E1_Alicante.csv ...


KeyboardInterrupt: 

In [ ]:
# Oredenador mas potente

"""
Notebook script (.py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación y combinado.
Los datasets para RF y XGB se guardan en formato Parquet (menor tamaño).
"""

import os
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
import joblib

# ---------------------- CONFIGURACION ----------------------

FILE_PATHS = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

OUTPUT_DIR = Path(r"/Volumes/copia seguridad1/Datos TFG").expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72

# Estrategias de imputación para columnas específicas (solo las que existen en los CSV)
IMPUTE_STRATEGIES = {
    'NO': 'time_interpolate',
    'NO2': 'time_interpolate',
    'NOx': 'time_interpolate',
    'O3': 'time_interpolate',
    'Veloc.': 'time_interpolate',
    'Direc.': 'circular',          # Se transformará a seno/coseno
    'Temp.': 'time_interpolate',
    'R.Sol.': 'time_interpolate',
    'Estacion': 'ffill',
    'Transecto': 'ffill',
    'Dist.': 'ffill',
    'Angulo': 'circular'
}

BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']   # Predictores numéricos base
CATEGORICAL = ['Estacion', 'Transecto']                              # Categóricas para one-hot
CIRCULAR_ORIG = ['Direc.', 'Angulo']                                 # Variables angulares (se transforman)
TARGET = 'O3'                                                         # Variable objetivo

# ---------------------- FUNCIONES DE PREPROCESADO ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    """Convierte una columna a datetime, la establece como índice y completa la serie horaria."""
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df = df.dropna(subset=[col])
        df = df.set_index(col)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No hay índice datetime válido")
    df = df.sort_index()
    # Eliminar duplicados temporales
    if not df.index.is_unique:
        df = df[~df.index.duplicated(keep='first')]
    # Crear índice horario completo
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    df = df.reindex(full_idx)
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    """Añade características cíclicas (seno/coseno) de hora, día, semana, mes y año."""
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

    years = year - year.min()
    denom = max(year.max() - year.min() + 1, 1)
    d['year_sin'] = np.sin(2 * np.pi * years / denom)
    d['year_cos'] = np.cos(2 * np.pi * years / denom)
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    """Convierte una serie de ángulos (grados) a seno y coseno."""
    radians = np.deg2rad(series)
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(radians),
        f'{prefix}_cos': np.cos(radians)
    }, index=series.index)


def impute_variable(df: pd.DataFrame, col: str, strategy: str) -> pd.Series:
    """
    Imputa una columna según la estrategia indicada.
    Estrategias: 'time_interpolate', 'ffill', 'median', 'constant'.
    """
    s = df[col]
    if strategy == 'time_interpolate':
        s_imputed = s.interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strategy == 'ffill':
        return s.ffill().bfill()
    elif strategy == 'median':
        return s.fillna(s.median())
    elif strategy == 'constant':
        return s.fillna(0)
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")


def preprocess_single_csv(path: str) -> pd.DataFrame:
    """Carga, limpia e imputa un único archivo CSV."""
    print(f"  Preprocesando {Path(path).name}...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # 1. Índice temporal completo
    df = ensure_datetime_index(df, 'datetime')

    # 2. Añadir características temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # 3. Transformar variables circulares a seno/coseno y eliminar las originales
    for circ in CIRCULAR_ORIG:
        if circ in df.columns:
            sincos = circular_to_sincos(df[circ], circ.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)
            df.drop(columns=[circ], inplace=True)

    # 4. Definir estrategias de imputación para todas las columnas presentes
    impute_map = {}
    for col in df.columns:
        if col in IMPUTE_STRATEGIES:
            strat = IMPUTE_STRATEGIES[col]
            if strat == 'circular':
                strat = 'time_interpolate'   # ya tenemos componentes _sin/_cos
            impute_map[col] = strat
        elif col in BASE_FEATURES or col == TARGET or col.endswith('_sin') or col.endswith('_cos'):
            impute_map[col] = 'time_interpolate'
        elif col in CATEGORICAL:
            impute_map[col] = 'ffill'
        else:
            impute_map[col] = IMPUTE_STRATEGIES.get(col, 'median')

    # 5. Aplicar imputación
    for col, strat in impute_map.items():
        if col in df.columns:
            df[col] = impute_variable(df, col, strat)

    # 6. Garantizar que columnas base y target no tengan NaNs
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return df


# ---------------------- VENTANAS DESLIZANTES ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: Optional[List[str]] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Genera ventanas deslizantes de entrada-salida.

    - Si model_type es 'rf' o 'xgb': aplana las ventanas y devuelve DataFrames (2D).
    - Si model_type es 'rnn' o 'lstm': devuelve arrays 3D (muestras, tiempo, características).

    features: lista de nombres de columnas a usar como predictores.
              Si es None, se toman todas las columnas numéricas del dataframe.
    """
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns.tolist()
        if target_col in features:
            features.remove(target_col)

    features = [f for f in features if f in df.columns]

    X_list, y_list, idxs = [], [], []
    total_len = len(df)

    for start in range(total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours

        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values

        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue

        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        X_flat = np.array(X_list)                     # (n, t, f)
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)                       # (n, output_hours)

        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d, index=idxs)
        y_df = pd.DataFrame(y_2d, columns=y_cols, index=idxs)
        return X_df, y_df
    else:
        return np.array(X_list), np.array(y_list)


# ---------------------- FUNCIÓN PRINCIPAL ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    """
    Procesa todos los archivos, guarda datasets individuales y combinados.
    RF y XGB se guardan en formato Parquet (comprimido con snappy).
    RNN/LSTM se guardan en formato .npz.
    """
    all_rf_X = []
    all_rf_y = []
    all_xgb_X = []
    all_xgb_y = []
    all_rnn_X = []
    all_rnn_y = []

    for p in file_paths:
        p = str(Path(p).expanduser())
        station_name = Path(p).stem
        print(f"\nProcesando estación: {station_name}")

        df = preprocess_single_csv(p)

        # ------------------------------------------------------------
        # 1. Datasets para RF y XGB (con one-hot encoding)
        # ------------------------------------------------------------
        cat_cols = [c for c in CATEGORICAL if c in df.columns]
        df_tab = pd.get_dummies(df, columns=cat_cols)

        feature_cols_tab = df_tab.select_dtypes(include=[np.number]).columns.tolist()
        if TARGET in feature_cols_tab:
            feature_cols_tab.remove(TARGET)

        X_rf, y_rf = make_sliding_windows(df_tab,
                                          features=feature_cols_tab,
                                          target_col=TARGET,
                                          model_type='rf')

        station_dir = output_dir / 'per_station' / station_name
        station_dir.mkdir(parents=True, exist_ok=True)

        # Guardar en Parquet (con compresión snappy)
        X_rf.to_parquet(station_dir / 'rf_X.parquet', compression='snappy', index=True)
        y_rf.to_parquet(station_dir / 'rf_y.parquet', compression='snappy', index=True)

        all_rf_X.append(X_rf)
        all_rf_y.append(y_rf)

        X_xgb, y_xgb = make_sliding_windows(df_tab,
                                            features=feature_cols_tab,
                                            target_col=TARGET,
                                            model_type='xgb')
        X_xgb.to_parquet(station_dir / 'xgb_X.parquet', compression='snappy', index=True)
        y_xgb.to_parquet(station_dir / 'xgb_y.parquet', compression='snappy', index=True)
        all_xgb_X.append(X_xgb)
        all_xgb_y.append(y_xgb)

        # ------------------------------------------------------------
        # 2. Datasets para RNN/LSTM (sin one-hot)
        # ------------------------------------------------------------
        feature_cols_rnn = df.select_dtypes(include=[np.number]).columns.tolist()
        if TARGET in feature_cols_rnn:
            feature_cols_rnn.remove(TARGET)

        X_rnn, y_rnn = make_sliding_windows(df,
                                            features=feature_cols_rnn,
                                            target_col=TARGET,
                                            model_type='rnn')
        np.savez_compressed(station_dir / 'rnn_lstm_data.npz', X=X_rnn, y=y_rnn)
        all_rnn_X.append(X_rnn)
        all_rnn_y.append(y_rnn)

        print(f"  Ventanas generadas: RF={len(X_rf)}, XGB={len(X_xgb)}, RNN={len(X_rnn)}")

    # ------------------------------------------------------------
    # 3. Datasets combinados
    # ------------------------------------------------------------
    print("\nGenerando datasets combinados...")
    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    if all_rf_X:
        combined_rf_X = pd.concat(all_rf_X, axis=0)
        combined_rf_y = pd.concat(all_rf_y, axis=0)
        combined_rf_X.to_parquet(combined_dir / 'rf_X.parquet', compression='snappy', index=True)
        combined_rf_y.to_parquet(combined_dir / 'rf_y.parquet', compression='snappy', index=True)

    if all_xgb_X:
        combined_xgb_X = pd.concat(all_xgb_X, axis=0)
        combined_xgb_y = pd.concat(all_xgb_y, axis=0)
        combined_xgb_X.to_parquet(combined_dir / 'xgb_X.parquet', compression='snappy', index=True)
        combined_xgb_y.to_parquet(combined_dir / 'xgb_y.parquet', compression='snappy', index=True)

    if all_rnn_X:
        combined_rnn_X = np.concatenate(all_rnn_X, axis=0)
        combined_rnn_y = np.concatenate(all_rnn_y, axis=0)
        np.savez_compressed(combined_dir / 'rnn_lstm_data.npz',
                            X=combined_rnn_X, y=combined_rnn_y)

    print(f"\n¡Proceso completado! Datos guardados en: {output_dir}")
    print("Los archivos de RF y XGB están en formato Parquet (comprimido).")


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)


Procesando estación: T1_E1_Alicante
  Preprocesando T1_E1_Alicante.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T1_E2_Elda
  Preprocesando T1_E2_Elda.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T2_E1_Elche
  Preprocesando T2_E1_Elche.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T2_E2_Elda
  Preprocesando T2_E2_Elda.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T3_E1_Valencia
  Preprocesando T3_E1_Valencia.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T3_E2_Buñol
  Preprocesando T3_E2_Buñol.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T4_E1_Valencia
  Preprocesando T4_E1_Valencia.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T4_E2_Villar_Arzobispo
  Preprocesando T4_E2_Villar_Arzobispo.csv...
  Ventanas generadas

: 

In [1]:
# Mismas columnas pero con un ordenador mas potente, para evitar problemas de memoria al generar los combinados por streaming y con memmap para RNN/LSTM.

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Script optimizado para generar datasets finales (RF, XGBoost, RNN, LSTM)
- Guarda RF/XGB por estación en Parquet (snappy) y combined por streaming (pyarrow).
- Guarda RNN/LSTM por estación en .npz y combined mediante numpy.memmap.

Corregido para unificar columnas entre estaciones:
   - Primera pasada: recopilar todos los nombres de columnas dummy y numéricas.
   - Segunda pasada: alinear cada DataFrame a esos conjuntos comunes.
"""
import sys
import os
import traceback
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Set

import numpy as np
import pandas as pd

# Optional helpers
try:
    from tqdm import tqdm
    TQDM = True
except ImportError:
    TQDM = False

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PYARROW = True
except ImportError:
    PYARROW = False

# For memory info (optional)
try:
    import psutil
    PSUTIL = True
except ImportError:
    PSUTIL = False

# ---------------------- CONFIGURACIÓN: ajústala ----------------------
FILE_PATHS = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
OUTPUT_DIR = Path(r"/Volumes/copia seguridad1/Datos TFG").expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72

IMPUTE_STRATEGIES = {
    'NO': 'time_interpolate',
    'NO2': 'time_interpolate',
    'NOx': 'time_interpolate',
    'O3': 'time_interpolate',
    'Veloc.': 'time_interpolate',
    'Direc.': 'circular',
    'Temp.': 'time_interpolate',
    'R.Sol.': 'time_interpolate',
    'Estacion': 'ffill',
    'Transecto': 'ffill',
    'Dist.': 'ffill',
    'Angulo': 'circular'
}
BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']
CATEGORICAL = ['Estacion', 'Transecto']
CIRCULAR_ORIG = ['Direc.', 'Angulo']
TARGET = 'O3'

# Umbrales de seguridad
MAX_REINDEX_ROWS = 5_000_000         # evitar reindexaciones monstruosas
MAX_SPAN_HOURS = 3_000_000          # umbral conservador de rango temporal (≈342 años)

# ---------------------- UTILIDADES ----------------------

def log(msg: str):
    print(msg, flush=True)

def human_size(nbytes: int) -> str:
    for unit in ['B','KB','MB','GB','TB']:
        if abs(nbytes) < 1024.0:
            return f"{nbytes:3.1f}{unit}"
        nbytes /= 1024.0
    return f"{nbytes:.1f}PB"

def memory_info() -> str:
    if PSUTIL:
        mem = psutil.virtual_memory()
        return f"mem: {human_size(mem.used)}/{human_size(mem.total)} ({mem.percent}%)"
    return "mem: psutil no instalado"

# ---------------------- PREPROCESADO ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df = df.dropna(subset=[col])
        df = df.set_index(col)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No hay índice datetime válido")
    df = df.sort_index()
    if not df.index.is_unique:
        df = df[~df.index.duplicated(keep='first')]

    span_hours = int((df.index.max() - df.index.min()) / np.timedelta64(1, 'h')) + 1
    if span_hours > MAX_SPAN_HOURS:
        raise ValueError(f"Rango temporal enorme: {span_hours} horas. Revisa el CSV (fechas erróneas).")

    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    if len(full_idx) - len(df) > MAX_REINDEX_ROWS:
        raise ValueError("La reindexación agregaría demasiadas filas vacías (posible fechas erróneas).")

    df = df.reindex(full_idx)
    return df

def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)
    years = year - year.min()
    denom = max(year.max() - year.min() + 1, 1)
    d['year_sin'] = np.sin(2 * np.pi * years / denom)
    d['year_cos'] = np.cos(2 * np.pi * years / denom)
    return d

def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    radians = np.deg2rad(series)
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(radians),
        f'{prefix}_cos': np.cos(radians)
    }, index=series.index)

def impute_variable(df: pd.DataFrame, col: str, strategy: str) -> pd.Series:
    s = df[col]
    if strategy == 'time_interpolate':
        s_imputed = s.interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strategy == 'ffill':
        return s.ffill().bfill()
    elif strategy == 'median':
        return s.fillna(s.median())
    elif strategy == 'constant':
        return s.fillna(0)
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric types to reduce memory (in-place copy)."""
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col].dtype):
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif pd.api.types.is_integer_dtype(df[col].dtype):
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

def preprocess_single_csv(path: str) -> pd.DataFrame:
    """
    Lectura robusta e imputación.
    """
    p = Path(path)
    log(f"  Leyendo (primeras 5k filas) {p.name} ...")
    try:
        df = pd.read_csv(p, parse_dates=['datetime'], low_memory=False)
    except (ValueError, KeyError):
        # no tiene 'datetime' en header o error; leer sin parse y convertir luego
        df = pd.read_csv(p, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    df = ensure_datetime_index(df, 'datetime')
    df = add_cyclical_datetime_features(df)
    for circ in CIRCULAR_ORIG:
        if circ in df.columns:
            sincos = circular_to_sincos(df[circ], circ.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)
            df.drop(columns=[circ], inplace=True)

    # map de imputación
    impute_map = {}
    for col in df.columns:
        if col in IMPUTE_STRATEGIES:
            strat = IMPUTE_STRATEGIES[col]
            if strat == 'circular':
                strat = 'time_interpolate'
            impute_map[col] = strat
        elif col in BASE_FEATURES or col == TARGET or col.endswith('_sin') or col.endswith('_cos'):
            impute_map[col] = 'time_interpolate'
        elif col in CATEGORICAL:
            impute_map[col] = 'ffill'
        else:
            impute_map[col] = IMPUTE_STRATEGIES.get(col, 'median')

    for col, strat in impute_map.items():
        if col in df.columns:
            df[col] = impute_variable(df, col, strat)

    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    # reducir memoria donde sea posible
    df = reduce_mem_usage(df)
    return df

# ---------------------- VENTANAS DESLIZANTES ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: Optional[List[str]] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf') -> Tuple:
    """
    Devuelve:
      - Para 'rf'/'xgb': (X_df, y_df) DataFrames
      - Para 'rnn'/'lstm': (X_array, y_array) numpy arrays (3D / 2D)
    Implementado de forma eficiente.
    """
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns.tolist()
        if target_col in features:
            features.remove(target_col)
    features = [f for f in features if f in df.columns]

    total_len = len(df)
    if total_len < input_hours + output_hours:
        # estructuras vacías
        if model_type in ['rf','xgb']:
            return pd.DataFrame(), pd.DataFrame()
        else:
            return np.empty((0, input_hours, len(features))), np.empty((0, output_hours))

    # Pre-allocación aproximada: no sabemos cuántas ventanas válidas (sin NaNs) habrá;
    # iremos acumulando por bloques pequeños para evitar listas enormes.
    X_blocks = []
    y_blocks = []
    idxs = []

    for start in range(total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours
        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        X_blocks.append(Xw)
        y_blocks.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        if len(X_blocks) == 0:
            return pd.DataFrame(), pd.DataFrame()
        X_flat = np.array(X_blocks)  # (n, t, f)
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_blocks)
        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d, index=idxs)
        y_df = pd.DataFrame(y_2d, columns=y_cols, index=idxs)
        return X_df, y_df
    else:
        return np.array(X_blocks), np.array(y_blocks)

# ---------------------- FUNCIONES PARA COMBINAR ----------------------

def write_parquet_streaming(df: pd.DataFrame, out_path: Path, parquet_schema: Optional[pa.Schema] = None, compression: str = 'snappy'):
    """
    Escribe (o anexa) un DataFrame a out_path usando pyarrow ParquetWriter.
    Si out_path existe, abre en modo append escribiendo el fragmento nuevo.
    """
    if not PYARROW:
        raise RuntimeError("pyarrow es necesario para escritura por streaming. Instálalo: pip install pyarrow")

    table = pa.Table.from_pandas(df)
    if not out_path.exists():
        writer = pq.ParquetWriter(out_path, table.schema, use_dictionary=True)
        writer.write_table(table)
        writer.close()
    else:
        existing = pq.ParquetFile(str(out_path))
        existing_schema = existing.schema.to_arrow_schema()
        # if schemas differ, try to unify by casting columns present in df
        table = table.cast(existing_schema, safe=False)
        with pq.ParquetWriter(out_path, existing_schema, use_dictionary=True) as writer:
            writer.write_table(table)

# ---------------------- MAIN: procesamiento por estación ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    log_dir = output_dir / 'logs'
    per_station_dir = output_dir / 'per_station'
    combined_dir = output_dir / 'combined'
    log_dir.mkdir(parents=True, exist_ok=True)
    per_station_dir.mkdir(parents=True, exist_ok=True)
    combined_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------------------------------------------
    # PRIMERA PASADA: recopilar todas las posibles columnas dummy y numéricas
    # -----------------------------------------------------------------
    log("=== PRIMERA PASADA: recopilando nombres de columnas ===")
    all_dummy_columns: Set[str] = set()
    all_numeric_columns: Set[str] = set()

    iterator1 = file_paths if not TQDM else tqdm(file_paths, desc="Primera pasada")
    for p in iterator1:
        p = str(Path(p).expanduser())
        try:
            df = preprocess_single_csv(p)

            # Para RF/XGB: get_dummies
            cat_cols = [c for c in CATEGORICAL if c in df.columns]
            df_tab = pd.get_dummies(df, columns=cat_cols)
            num_cols_tab = df_tab.select_dtypes(include=[np.number]).columns.tolist()
            all_dummy_columns.update(num_cols_tab)

            # Para RNN/LSTM: columnas numéricas (excluyendo target)
            num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if TARGET in num_cols:
                num_cols.remove(TARGET)
            all_numeric_columns.update(num_cols)

        except Exception as e:
            log(f"  Error en primera pasada con {Path(p).stem}: {e}")

    # Convertir a listas ordenadas para consistencia
    all_dummy_columns = sorted(all_dummy_columns)
    all_numeric_columns = sorted(all_numeric_columns)
    log(f"Columnas dummy totales: {len(all_dummy_columns)}")
    log(f"Columnas numéricas totales: {len(all_numeric_columns)}")

    # -----------------------------------------------------------------
    # SEGUNDA PASADA: procesar cada estación alineando a los conjuntos comunes
    # -----------------------------------------------------------------
    log("\n=== SEGUNDA PASADA: generando ventanas ===")

    # Registros para combinar RNN después
    rnn_counts = []  # list of (station_dir / filename, n_samples, input_hours, output_hours, feat_count)

    iterator2 = file_paths if not TQDM else tqdm(file_paths, desc="Segunda pasada")
    for p in iterator2:
        p = str(Path(p).expanduser())
        station_name = Path(p).stem
        station_dir = per_station_dir / station_name
        station_dir.mkdir(parents=True, exist_ok=True)
        try:
            log(f"\nProcesando estación: {station_name}  ({memory_info()})")
            df = preprocess_single_csv(p)

            # ----- RF & XGB (one-hot para categóricas) -----
            cat_cols = [c for c in CATEGORICAL if c in df.columns]
            df_tab = pd.get_dummies(df, columns=cat_cols)

            # Alinear con todas las columnas dummy posibles
            for col in all_dummy_columns:
                if col not in df_tab.columns:
                    df_tab[col] = 0.0
            df_tab = df_tab[all_dummy_columns]   # orden fijo

            feature_cols_tab = all_dummy_columns   # ya son todas las numéricas tras get_dummies

            # RF
            X_rf, y_rf = make_sliding_windows(df_tab,
                                              features=feature_cols_tab,
                                              target_col=TARGET,
                                              model_type='rf')
            if not X_rf.empty:
                X_rf.to_parquet(station_dir / 'rf_X.parquet', compression='snappy', index=True, engine='pyarrow')
                y_rf.to_parquet(station_dir / 'rf_y.parquet', compression='snappy', index=True, engine='pyarrow')
                log(f"  Guardado RF (parquet) -> {station_dir / 'rf_X.parquet'}")

            # XGB (mismo contenido que RF en este flujo)
            # Podemos reutilizar los mismos DataFrames (son iguales)
            if not X_rf.empty:
                X_rf.to_parquet(station_dir / 'xgb_X.parquet', compression='snappy', index=True, engine='pyarrow')
                y_rf.to_parquet(station_dir / 'xgb_y.parquet', compression='snappy', index=True, engine='pyarrow')
                log(f"  Guardado XGB (parquet) -> {station_dir / 'xgb_X.parquet'}")

            # ----- RNN/LSTM (sin one-hot) -----
            # Alinear con todas las columnas numéricas posibles
            for col in all_numeric_columns:
                if col not in df.columns:
                    df[col] = 0.0   # o usar mediana si se prefiere
            df_num = df[all_numeric_columns]

            X_rnn, y_rnn = make_sliding_windows(df_num,
                                                features=all_numeric_columns,
                                                target_col=TARGET,
                                                model_type='rnn')

            np.savez_compressed(station_dir / 'rnn_lstm_data.npz', X=X_rnn, y=y_rnn)
            n_samp = X_rnn.shape[0] if X_rnn.size else 0
            rnn_counts.append((station_dir / 'rnn_lstm_data.npz', n_samp, X_rnn.shape[1] if X_rnn.size else 0, y_rnn.shape[1] if y_rnn.size else 0, len(all_numeric_columns)))
            log(f"  Guardado RNN npz -> {station_dir / 'rnn_lstm_data.npz'} (n_samples={n_samp})")

            log(f"  Ventanas: RF={len(X_rf) if hasattr(X_rf,'shape') else 0}, XGB={len(X_rf) if hasattr(X_rf,'shape') else 0}, RNN={n_samp}")

        except Exception as e:
            logf = log_dir / f"{station_name}_error.log"
            with open(logf, 'w', encoding='utf-8') as fh:
                fh.write(f"Error procesando {station_name} ({p}):\n")
                fh.write(traceback.format_exc())
            log(f"  ¡Error procesando {station_name}! Ver {logf}")

    # -----------------------------------------------------------------
    # COMBINAR PARQUET PARA RF/XGB (streaming)
    # -----------------------------------------------------------------
    if not PYARROW:
        log("\npyarrow no disponible: no se pueden crear combinados por streaming.")
    else:
        log("\n=== COMBINANDO PARQUET ===")
        def combine_parquets(pattern: str, out_name: str):
            parts = list(per_station_dir.rglob(pattern))
            if not parts:
                log(f"  No se encontraron archivos {pattern}")
                return
            out_path = combined_dir / out_name
            # Si existe, lo borramos para reescribir de cero
            if out_path.exists():
                out_path.unlink()
            writer = None
            for i, part in enumerate(parts):
                try:
                    df_part = pd.read_parquet(part, engine='pyarrow')
                    table = pa.Table.from_pandas(df_part)
                    if writer is None:
                        writer = pq.ParquetWriter(out_path, table.schema, use_dictionary=True)
                    writer.write_table(table)
                except Exception as e:
                    log(f"  Error añadiendo {part} al combinado: {e}")
            if writer:
                writer.close()
                log(f"  Combined escrito: {out_path} (parts={len(parts)})")

        combine_parquets("rf_X.parquet", "rf_X.parquet")
        combine_parquets("rf_y.parquet", "rf_y.parquet")
        combine_parquets("xgb_X.parquet", "xgb_X.parquet")
        combine_parquets("xgb_y.parquet", "xgb_y.parquet")

    # -----------------------------------------------------------------
    # COMBINAR RNN/LSTM (.npz) EN UN ÚNICO .npz USANDO MEMMAP
    # -----------------------------------------------------------------
    log("\n=== COMBINANDO RNN/LSTM ===")
    total_samples = 0
    max_in = 0
    max_out = 0
    feat_count = len(all_numeric_columns)   # ya unificado
    valid_npzs = []
    for npz_path, n_samples, in_h, out_h, fcount in rnn_counts:
        if n_samples > 0:
            valid_npzs.append((npz_path, n_samples, in_h, out_h))
            total_samples += n_samples
            max_in = max(max_in, in_h)
            max_out = max(max_out, out_h)

    if total_samples == 0:
        log("No hay muestras RNN para combinar.")
    else:
        log(f"Combinando RNN/LSTM: total_samples={total_samples}, input_hours={max_in}, output_hours={max_out}, feat_count={feat_count}")

        # Crear memmap para X (float32) y archivo final
        combined_X_path = combined_dir / 'rnn_X.dat'
        combined_y_path = combined_dir / 'rnn_y.dat'
        if combined_X_path.exists(): combined_X_path.unlink()
        if combined_y_path.exists(): combined_y_path.unlink()

        X_mem = np.memmap(str(combined_X_path), dtype='float32', mode='w+', shape=(total_samples, max_in, feat_count))
        y_mem = np.memmap(str(combined_y_path), dtype='float32', mode='w+', shape=(total_samples, max_out))

        write_ptr = 0
        for npz_path, n_samples, in_h, out_h in valid_npzs:
            try:
                with np.load(npz_path) as data:
                    X_part = data['X']  # shape (n_samples, in_h, feat_count) ya unificado
                    y_part = data['y']  # shape (n_samples, out_h)

                    # Asegurar dimensiones (padding si alguna estación tiene menos horas)
                    if X_part.shape[1] < max_in:
                        pad_width = max_in - X_part.shape[1]
                        pad_shape = (X_part.shape[0], pad_width, X_part.shape[2])
                        X_pad = np.zeros(pad_shape, dtype=X_part.dtype)
                        X_part = np.concatenate([X_pad, X_part], axis=1)
                    if y_part.shape[1] < max_out:
                        pad_width = max_out - y_part.shape[1]
                        y_pad = np.zeros((y_part.shape[0], pad_width), dtype=y_part.dtype)
                        y_part = np.concatenate([y_part, y_pad], axis=1)

                    X_mem[write_ptr:write_ptr + X_part.shape[0], :, :] = X_part.astype('float32')
                    y_mem[write_ptr:write_ptr + y_part.shape[0], :] = y_part.astype('float32')
                    write_ptr += X_part.shape[0]
                    log(f"  Copiadas {X_part.shape[0]} muestras desde {npz_path.name} (ptr={write_ptr}/{total_samples})")
            except Exception as e:
                log(f"  Error leyendo {npz_path}: {e}")

        X_mem.flush()
        y_mem.flush()

        # Guardar a npz comprimido final usando archivos npy temporales
        tmp_X_npy = combined_dir / 'tmp_rnn_X.npy'
        tmp_y_npy = combined_dir / 'tmp_rnn_y.npy'
        combined_npz_path = combined_dir / 'rnn_lstm_data_combined.npz'

        for q in (tmp_X_npy, tmp_y_npy, combined_npz_path):
            if q.exists():
                q.unlink()

        np.save(tmp_X_npy, X_mem)
        np.save(tmp_y_npy, y_mem)

        try:
            X_comb = np.load(tmp_X_npy, mmap_mode='r')
            y_comb = np.load(tmp_y_npy, mmap_mode='r')
            np.savez_compressed(combined_npz_path, X=X_comb, y=y_comb)
            log(f"  Combined RNN npz guardado: {combined_npz_path} (samples={total_samples})")
            # limpiar temporales
            tmp_X_npy.unlink()
            tmp_y_npy.unlink()
            combined_X_path.unlink()
            combined_y_path.unlink()
        except Exception as e:
            log(f"  No se pudo crear .npz comprimido final (memoria insuficiente). Los archivos .npy están en {combined_dir}. Puedes convertirlos manualmente.")

    log("\n¡Proceso completado! Archivos en: " + str(output_dir))

# ---------------------- ENTRYPOINT ----------------------

if __name__ == '__main__':
    if not PYARROW:
        log("Aviso: pyarrow no está instalado. Instálalo para combinados Parquet por streaming: pip install pyarrow")
    try:
        prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)
    except Exception:
        traceback.print_exc()
        sys.exit(1)

=== PRIMERA PASADA: recopilando nombres de columnas ===
  Leyendo (primeras 5k filas) T1_E1_Alicante.csv ...
  Leyendo (primeras 5k filas) T1_E2_Elda.csv ...
  Leyendo (primeras 5k filas) T2_E1_Elche.csv ...
  Leyendo (primeras 5k filas) T2_E2_Elda.csv ...
  Leyendo (primeras 5k filas) T3_E1_Valencia.csv ...
  Leyendo (primeras 5k filas) T3_E2_Buñol.csv ...
  Leyendo (primeras 5k filas) T4_E1_Valencia.csv ...
  Leyendo (primeras 5k filas) T4_E2_Villar_Arzobispo.csv ...
  Leyendo (primeras 5k filas) T6_E1_Castellon.csv ...
  Leyendo (primeras 5k filas) T6_E2_Onda.csv ...
  Leyendo (primeras 5k filas) T7_E1_Sant_Jordi.csv ...
  Leyendo (primeras 5k filas) T7_E2_Coratxa.csv ...
  Leyendo (primeras 5k filas) T7_E3_Zorita.csv ...
  Leyendo (primeras 5k filas) T8_E1_Sant_Jordi.csv ...
  Leyendo (primeras 5k filas) T8_E2_Morella.csv ...
  Leyendo (primeras 5k filas) T8_E3_Zorita.csv ...
Columnas dummy totales: 23
Columnas numéricas totales: 22

=== SEGUNDA PASADA: generando ventanas ===

Proc

In [ ]:
"""
Script para entrenar y evaluar los 4 modelos (RF, XGB, RNN, LSTM)
utilizando los datasets generados por 'generar_datasets.py' (ahora en formato Parquet).

Se asume que los datos se encuentran en la estructura de carpetas:
  ~/Documents/GitHub/TFGFinal/Datos TFG/DatasetsFinales/
    per_station/
        <nombre_estacion>/
            rf_X.parquet, rf_y.parquet, xgb_X.parquet, xgb_y.parquet, rnn_lstm_data.npz
    combined/
            rf_X.parquet, rf_y.parquet, xgb_X.parquet, xgb_y.parquet, rnn_lstm_data.npz

El script permite elegir:
  - Modo 'station': procesa una estación concreta (por nombre).
  - Modo 'combined': procesa el dataset combinado.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, Optional, Union

# Modelos de sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# TensorFlow / Keras para RNN y LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuración de rutas
BASE_DIR = Path(r"/Volumes/copia seguridad1/Datos TFG").expanduser()
OUTPUT_DIR = BASE_DIR / "resultados_entrenamiento"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parámetros globales
INPUT_HOURS = 72
OUTPUT_HOURS = 72
RANDOM_STATE = 42
TEST_SIZE = 0.15          # 15% para test
VAL_SIZE = 0.15           # 15% para validación (sobre el resto después de test)
SCALE_FEATURES = True      # Escalar características (importante para RNN/LSTM y puede ayudar a RF/XGB)

# Selección de modo y estación (cambiar según necesidad)
MODE = 'station'          # 'station' o 'combined'
STATION_NAME = 'T1_E1_Alicante'   # Solo si MODE == 'station'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def load_rf_xgb_data(mode: str, station: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Carga los datos de RF o XGB desde archivos Parquet.
    Restaura el índice datetime que se guardó como columna.
    """
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    X = pd.read_parquet(data_dir / 'rf_X.parquet')
    y = pd.read_parquet(data_dir / 'rf_y.parquet')

    # El índice se guardó como columna (por defecto se llama 'index' o la primera columna sin nombre)
    # Identificamos la columna que contiene las fechas (será de tipo datetime)
    for col in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[col]):
            X = X.set_index(col)
            break
    else:
        # Si no hay columna datetime, intentamos con la primera columna si parece una fecha
        first_col = X.columns[0]
        try:
            X[first_col] = pd.to_datetime(X[first_col])
            X = X.set_index(first_col)
        except:
            pass  # dejamos el índice numérico por defecto

    for col in y.columns:
        if pd.api.types.is_datetime64_any_dtype(y[col]):
            y = y.set_index(col)
            break
    else:
        first_col = y.columns[0]
        try:
            y[first_col] = pd.to_datetime(y[first_col])
            y = y.set_index(first_col)
        except:
            pass

    # Asegurar que el índice es datetime
    if not isinstance(X.index, pd.DatetimeIndex):
        X.index = pd.to_datetime(X.index)
    if not isinstance(y.index, pd.DatetimeIndex):
        y.index = pd.to_datetime(y.index)

    return X, y

def load_rnn_lstm_data(mode: str, station: Optional[str] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Carga los datos de RNN/LSTM (formato npz con arrays 3D)."""
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    data = np.load(data_dir / 'rnn_lstm_data.npz')
    return data['X'], data['y']

def temporal_train_val_test_split(X, y, val_size=0.15, test_size=0.15, random_state=None):
    """
    Divide los datos en train/val/test respetando el orden temporal.
    Asume que X e y están indexados por tiempo (para pandas DataFrame) o
    que las muestras están en orden cronológico (para arrays numpy).
    """
    n = len(X)
    test_cut = int(n * (1 - test_size))
    val_cut = int(test_cut * (1 - val_size))

    if isinstance(X, pd.DataFrame):
        # Para pandas, mantener índices
        X_train, y_train = X.iloc[:val_cut], y.iloc[:val_cut]
        X_val, y_val = X.iloc[val_cut:test_cut], y.iloc[val_cut:test_cut]
        X_test, y_test = X.iloc[test_cut:], y.iloc[test_cut:]
    else:
        # Para numpy arrays
        X_train, y_train = X[:val_cut], y[:val_cut]
        X_val, y_val = X[val_cut:test_cut], y[val_cut:test_cut]
        X_test, y_test = X[test_cut:], y[test_cut:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def scale_features(X_train, X_val, X_test, feature_names=None):
    """
    Escala las características usando StandardScaler.
    Devuelve los conjuntos escalados y el scaler ajustado.
    """
    scaler = StandardScaler()
    # Ajustar con entrenamiento
    if isinstance(X_train, pd.DataFrame):
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        # Convertir de vuelta a DataFrame si se desea (opcional)
        X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
        X_val_scaled = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
        X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
    else:
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

def evaluate_model(y_true, y_pred, model_name, dataset_type='test'):
    """Calcula métricas y las imprime. Devuelve diccionario con las métricas."""
    mae = mean_absolute_error(y_true, y_pred, multioutput='uniform_average')
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='uniform_average'))
    r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
    print(f"{model_name} - {dataset_type} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

def plot_predictions(y_true, y_pred, model_name, num_samples=5):
    """Dibuja las primeras num_samples predicciones vs reales (para una hora específica, p.ej. horizonte 24)."""
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))
    if num_samples == 1:
        axes = [axes]
    for i in range(num_samples):
        ax = axes[i]
        ax.plot(y_true[i], label='Real', marker='o', linestyle='-')
        ax.plot(y_pred[i], label='Predicho', marker='x', linestyle='--')
        ax.set_title(f'{model_name} - Muestra {i+1}')
        ax.set_xlabel('Horas predicción')
        ax.set_ylabel('O3')
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_predicciones_ejemplo.png')
    plt.show()

# ---------------------- ENTRENAMIENTO DE MODELOS ----------------------

def train_random_forest(X_train, y_train, X_val, y_val, X_test, y_test):
    """Entrena Random Forest multi-output y evalúa."""
    print("\n--- Entrenando Random Forest ---")
    model = RandomForestRegressor(n_estimators=100, max_depth=20,
                                  random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train.values, y_pred_train, "RF", "train")
    metrics_val = evaluate_model(y_val.values, y_pred_val, "RF", "val")
    metrics_test = evaluate_model(y_test.values, y_pred_test, "RF", "test")

    import joblib
    joblib.dump(model, OUTPUT_DIR / 'random_forest_model.pkl')
    print("Modelo RF guardado en:", OUTPUT_DIR / 'random_forest_model.pkl')

    plot_predictions(y_test.values[:5], y_pred_test[:5], "Random Forest")
    return model, metrics_test

def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test):
    """Entrena XGBoost multi-output y evalúa."""
    print("\n--- Entrenando XGBoost ---")
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100,
                             max_depth=6, learning_rate=0.1,
                             random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train.values, y_pred_train, "XGB", "train")
    metrics_val = evaluate_model(y_val.values, y_pred_val, "XGB", "val")
    metrics_test = evaluate_model(y_test.values, y_pred_test, "XGB", "test")

    import joblib
    joblib.dump(model, OUTPUT_DIR / 'xgboost_model.pkl')
    print("Modelo XGB guardado en:", OUTPUT_DIR / 'xgboost_model.pkl')

    plot_predictions(y_test.values[:5], y_pred_test[:5], "XGBoost")
    return model, metrics_test

def build_rnn_model(input_shape, output_steps):
    model = Sequential([
        SimpleRNN(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        SimpleRNN(64, return_sequences=False),
        Dropout(0.2),
        Dense(output_steps)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(input_shape, output_steps):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(output_steps)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def train_rnn_lstm(model_type, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\n--- Entrenando {model_type.upper()} ---")
    input_shape = (X_train.shape[1], X_train.shape[2])
    output_steps = y_train.shape[1]

    if model_type == 'rnn':
        model = build_rnn_model(input_shape, output_steps)
    else:
        model = build_lstm_model(input_shape, output_steps)

    model.summary()

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train, y_pred_train, model_type.upper(), "train")
    metrics_val = evaluate_model(y_val, y_pred_val, model_type.upper(), "val")
    metrics_test = evaluate_model(y_test, y_pred_test, model_type.upper(), "test")

    model.save(OUTPUT_DIR / f'{model_type}_model.h5')
    np.save(OUTPUT_DIR / f'{model_type}_history.npy', history.history)
    print(f"Modelo {model_type.upper()} guardado en:", OUTPUT_DIR / f'{model_type}_model.h5')

    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'{model_type.upper()} - Loss')
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history.history['mae'], label='train_mae')
    plt.plot(history.history['val_mae'], label='val_mae')
    plt.title(f'{model_type.upper()} - MAE')
    plt.legend()
    plt.savefig(OUTPUT_DIR / f'{model_type}_training_history.png')
    plt.show()

    plot_predictions(y_test[:5], y_pred_test[:5], model_type.upper())

    return model, metrics_test

# ---------------------- MAIN ----------------------

def main():
    print(f"Modo seleccionado: {MODE}")
    if MODE == 'station':
        print(f"Estación: {STATION_NAME}")

    # 1. Cargar datos para RF/XGB (desde Parquet)
    X_rf, y_rf = load_rf_xgb_data(MODE, STATION_NAME if MODE=='station' else None)
    print(f"Datos RF/XGB cargados: {X_rf.shape[0]} muestras, {X_rf.shape[1]} features (aplanadas)")

    # División temporal para RF/XGB
    (X_rf_train, y_rf_train), (X_rf_val, y_rf_val), (X_rf_test, y_rf_test) = \
        temporal_train_val_test_split(X_rf, y_rf, val_size=VAL_SIZE, test_size=TEST_SIZE)
    print(f"RF/XGB - Train: {len(X_rf_train)}, Val: {len(X_rf_val)}, Test: {len(X_rf_test)}")

    # Escalado opcional para RF/XGB
    if SCALE_FEATURES:
        X_rf_train, X_rf_val, X_rf_test, scaler_rf = scale_features(
            X_rf_train, X_rf_val, X_rf_test)
        import joblib
        joblib.dump(scaler_rf, OUTPUT_DIR / 'scaler_rf_xgb.pkl')
        print("Scaler para RF/XGB guardado.")

    # 2. Entrenar RF
    rf_model, rf_metrics = train_random_forest(
        X_rf_train, y_rf_train,
        X_rf_val, y_rf_val,
        X_rf_test, y_rf_test
    )

    # 3. Entrenar XGBoost
    xgb_model, xgb_metrics = train_xgboost(
        X_rf_train, y_rf_train,
        X_rf_val, y_rf_val,
        X_rf_test, y_rf_test
    )

    # 4. Cargar datos para RNN/LSTM (desde .npz)
    X_rnn, y_rnn = load_rnn_lstm_data(MODE, STATION_NAME if MODE=='station' else None)
    print(f"Datos RNN/LSTM cargados: {X_rnn.shape[0]} muestras, timesteps={X_rnn.shape[1]}, features={X_rnn.shape[2]}")

    (X_rnn_train, y_rnn_train), (X_rnn_val, y_rnn_val), (X_rnn_test, y_rnn_test) = \
        temporal_train_val_test_split(X_rnn, y_rnn, val_size=VAL_SIZE, test_size=TEST_SIZE)
    print(f"RNN/LSTM - Train: {len(X_rnn_train)}, Val: {len(X_rnn_val)}, Test: {len(X_rnn_test)}")

    # Escalado para RNN/LSTM
    if SCALE_FEATURES:
        original_shape = X_rnn_train.shape
        n_samples_train, n_timesteps, n_features = original_shape
        X_rnn_train_2d = X_rnn_train.reshape(-1, n_features)
        X_rnn_val_2d = X_rnn_val.reshape(-1, n_features)
        X_rnn_test_2d = X_rnn_test.reshape(-1, n_features)

        scaler_rnn = StandardScaler()
        X_rnn_train_2d = scaler_rnn.fit_transform(X_rnn_train_2d)
        X_rnn_val_2d = scaler_rnn.transform(X_rnn_val_2d)
        X_rnn_test_2d = scaler_rnn.transform(X_rnn_test_2d)

        X_rnn_train = X_rnn_train_2d.reshape(original_shape)
        X_rnn_val = X_rnn_val_2d.reshape(X_rnn_val.shape)
        X_rnn_test = X_rnn_test_2d.reshape(X_rnn_test.shape)

        import joblib
        joblib.dump(scaler_rnn, OUTPUT_DIR / 'scaler_rnn_lstm.pkl')
        print("Scaler para RNN/LSTM guardado.")

    # 5. Entrenar RNN
    rnn_model, rnn_metrics = train_rnn_lstm(
        'rnn',
        X_rnn_train, y_rnn_train,
        X_rnn_val, y_rnn_val,
        X_rnn_test, y_rnn_test
    )

    # 6. Entrenar LSTM
    lstm_model, lstm_metrics = train_rnn_lstm(
        'lstm',
        X_rnn_train, y_rnn_train,
        X_rnn_val, y_rnn_val,
        X_rnn_test, y_rnn_test
    )

    # 7. Comparativa final
    print("\n" + "="*50)
    print("RESUMEN DE MÉTRICAS EN TEST")
    print("="*50)
    comparativa = pd.DataFrame({
        'RF': rf_metrics,
        'XGB': xgb_metrics,
        'RNN': rnn_metrics,
        'LSTM': lstm_metrics
    }).T
    print(comparativa)
    comparativa.to_csv(OUTPUT_DIR / 'comparativa_metricas.csv')

    print(f"\nTodos los resultados guardados en: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
entrenar_modelos.py - Entrenamiento + búsqueda amplia de hiperparámetros
Entrena y evalúa RandomForest, XGBoost (MultiOutput via wrapper),
RNN y LSTM con búsquedas grandes de hiperparámetros (Randomized / Random Search).
Genera outputs en OUTPUT_DIR: modelos, scalers, logs de búsqueda y comparativa final.
"""
import os
import json
import time
import joblib
import random
import traceback
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ---------------------- CONFIGURACIÓN ----------------------
BASE_DIR = Path(r"/Volumes/copia seguridad1/Datos TFG").expanduser()
OUTPUT_DIR = BASE_DIR / "resultados_entrenamiento_hypersearch"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72
RANDOM_STATE = 42

TEST_SIZE = 0.15
VAL_SIZE = 0.15

# Hypersearch sizes (ajusta según tu hardware)
CV_SPLITS = 4                # TimeSeriesSplit folds for RF/XGB CV
N_ITER_RF = 60               # RandomizedSearch iterations for RF (large)
N_ITER_XGB = 80              # RandomizedSearch iterations for XGB (larger)
N_TRIALS_RNN = 40            # Random trials for RNN
N_TRIALS_LSTM = 40          # Random trials for LSTM

MAX_JOBS = -1  # n_jobs for RandomizedSearchCV (-1 uses all cores)

# MODE: 'station' or 'combined'
MODE = 'station'
STATION_NAME = 'T1_E1_Alicante'

# Seed for reproducibility (numpy/tf/random)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# ---------------------- UTILIDADES ----------------------
def log(msg: str):
    print(msg, flush=True)

def evaluate_model(y_true, y_pred, model_name: str, dataset_type='test') -> Dict[str, float]:
    mae = mean_absolute_error(y_true, y_pred, multioutput='uniform_average')
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='uniform_average'))
    r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
    log(f"{model_name} - {dataset_type} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
    return {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2)}

def plot_predictions(y_true, y_pred, model_name, num_samples=5):
    nplot = min(num_samples, len(y_true))
    fig, axes = plt.subplots(nplot, 1, figsize=(12, 3*nplot))
    if nplot == 1:
        axes = [axes]
    for i in range(nplot):
        ax = axes[i]
        ax.plot(y_true[i], label='Real', marker='o', linestyle='-')
        ax.plot(y_pred[i], label='Predicho', marker='x', linestyle='--')
        ax.set_title(f'{model_name} - Muestra {i+1}')
        ax.set_xlabel('Horas predicción')
        ax.set_ylabel('O3')
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_predicciones_ejemplo.png')
    plt.close(fig)

# ---------------------- CARGA DATOS ----------------------
def load_rf_xgb_data(mode: str, station: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    X_path = data_dir / 'rf_X.parquet'
    y_path = data_dir / 'rf_y.parquet'
    if not X_path.exists() or not y_path.exists():
        raise FileNotFoundError(f"Archivos rf_X/rf_y no encontrados en {data_dir}")

    X = pd.read_parquet(X_path)
    y = pd.read_parquet(y_path)

    # Restaurar índice datetime si existe columna 'datetime'
    if 'datetime' in X.columns:
        X['datetime'] = pd.to_datetime(X['datetime'], errors='coerce')
        X = X.set_index('datetime')
    else:
        X.index = pd.to_datetime(X.index)

    if 'datetime' in y.columns:
        y['datetime'] = pd.to_datetime(y['datetime'], errors='coerce')
        y = y.set_index('datetime')
    else:
        y.index = pd.to_datetime(y.index)

    # Asegurar alineamiento de índices y longitudes
    if len(X) != len(y):
        # intentar alinear por índice: inner join
        common_idx = X.index.intersection(y.index)
        X = X.loc[common_idx]
        y = y.loc[common_idx]
    return X, y

def load_rnn_lstm_data(mode: str, station: Optional[str] = None) -> Tuple[np.ndarray, np.ndarray]:
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
        combined_file = data_dir / 'rnn_lstm_data_combined.npz'
        fallback = data_dir / 'rnn_lstm_data.npz'
        file = combined_file if combined_file.exists() else fallback
    else:
        data_dir = BASE_DIR / 'per_station' / station
        file = data_dir / 'rnn_lstm_data.npz'
    if not file.exists():
        raise FileNotFoundError(f"No se encontró {file}")
    data = np.load(file)
    X = data['X'].astype('float32')
    y = data['y'].astype('float32')
    return X, y

# ---------------------- SPLIT TEMPORAL ----------------------
def temporal_train_val_test_split(X, y, val_size=0.15, test_size=0.15):
    n = len(X)
    test_cut = int(n * (1 - test_size))
    val_cut = int(test_cut * (1 - val_size))
    if isinstance(X, pd.DataFrame):
        X_train, y_train = X.iloc[:val_cut], y.iloc[:val_cut]
        X_val, y_val = X.iloc[val_cut:test_cut], y.iloc[val_cut:test_cut]
        X_test, y_test = X.iloc[test_cut:], y.iloc[test_cut:]
    else:
        X_train, y_train = X[:val_cut], y[:val_cut]
        X_val, y_val = X[val_cut:test_cut], y[val_cut:test_cut]
        X_test, y_test = X[test_cut:], y[test_cut:]
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

# ---------------------- ESCALADO ----------------------
def scale_features(X_train, X_val, X_test):
    scaler = StandardScaler()
    if isinstance(X_train, pd.DataFrame):
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
        X_val_scaled = pd.DataFrame(scaler.transform(X_val), index=X_val.index, columns=X_val.columns)
        X_test_scaled = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)
    else:
        # for RNN: flatten time dimension, fit scaler, then reshape back
        n_train, t, f = X_train.shape
        X_train_2d = X_train.reshape(-1, f)
        X_val_2d = X_val.reshape(-1, f)
        X_test_2d = X_test.reshape(-1, f)
        X_train_scaled_2d = scaler.fit_transform(X_train_2d)
        X_val_scaled_2d = scaler.transform(X_val_2d)
        X_test_scaled_2d = scaler.transform(X_test_2d)
        X_train_scaled = X_train_scaled_2d.reshape(n_train, t, f)
        X_val_scaled = X_val_scaled_2d.reshape(X_val.shape)
        X_test_scaled = X_test_scaled_2d.reshape(X_test.shape)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

# ---------------------- HYPERSEARCH: RF / XGB (RandomizedSearchCV con TimeSeriesSplit) ----------------------
def randomized_search_rf(X_train, y_train):
    log("\nIniciando RandomizedSearch para RandomForest (esto puede tardar)...")
    tss = TimeSeriesSplit(n_splits=CV_SPLITS)
    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)  # n_jobs controlado por RandomizedSearchCV
    param_dist = {
        'n_estimators': [100, 200, 400, 800, 1200],
        'max_depth': [10, 20, 30, 40, None],
        'max_features': ['sqrt', 'log2', 0.2, 0.5, 0.8],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }
    # Multi-output: RF admite multi-output nativo
    rs = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=N_ITER_RF,
                            cv=tss, scoring='neg_mean_squared_error', n_jobs=MAX_JOBS,
                            random_state=RANDOM_STATE, verbose=2)
    start = time.time()
    rs.fit(X_train, y_train)
    elapsed = time.time() - start
    log(f"RandomForest RandomizedSearch completado en {elapsed/60:.2f} min. Mejor score (neg MSE): {rs.best_score_:.6f}")
    # Guardar resultados
    pd.DataFrame(rs.cv_results_).to_csv(OUTPUT_DIR / 'rf_cv_results.csv', index=False)
    joblib.dump(rs.best_estimator_, OUTPUT_DIR / 'rf_best_model.pkl')
    with open(OUTPUT_DIR / 'rf_best_params.json', 'w') as fh:
        json.dump(rs.best_params_, fh, indent=2)
    return rs.best_estimator_, rs

def randomized_search_xgb(X_train, y_train):
    log("\nIniciando RandomizedSearch para XGBoost (envolviendo en MultiOutputRegressor)...")
    tss = TimeSeriesSplit(n_splits=CV_SPLITS)
    base = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=1)
    model = MultiOutputRegressor(base, n_jobs=1)
    # parámetros para pasar al estimador dentro del wrapper: "estimator__param"
    param_dist = {
        'estimator__n_estimators': [100, 200, 400, 800],
        'estimator__max_depth': [3, 5, 7, 9, 12],
        'estimator__learning_rate': [0.01, 0.02, 0.05, 0.1, 0.2],
        'estimator__subsample': [0.6, 0.8, 1.0],
        'estimator__colsample_bytree': [0.5, 0.7, 1.0],
        'estimator__reg_alpha': [0, 0.1, 0.5, 1.0],
        'estimator__reg_lambda': [1.0, 5.0, 10.0]
    }
    rs = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=N_ITER_XGB,
                            cv=tss, scoring='neg_mean_squared_error', n_jobs=MAX_JOBS,
                            random_state=RANDOM_STATE, verbose=2)
    start = time.time()
    rs.fit(X_train, y_train)
    elapsed = time.time() - start
    log(f"XGBoost RandomizedSearch completado en {elapsed/60:.2f} min. Mejor score (neg MSE): {rs.best_score_:.6f}")
    pd.DataFrame(rs.cv_results_).to_csv(OUTPUT_DIR / 'xgb_cv_results.csv', index=False)
    joblib.dump(rs.best_estimator_, OUTPUT_DIR / 'xgb_best_model.pkl')
    # best_params_ will have keys with 'estimator__...' - guardamos tal cual
    with open(OUTPUT_DIR / 'xgb_best_params.json', 'w') as fh:
        json.dump(rs.best_params_, fh, indent=2)
    return rs.best_estimator_, rs

# ---------------------- HYPERSEARCH: RNN / LSTM (Random search manual) ----------------------
def build_rnn_model(input_shape: Tuple[int,int], units1: int, units2: int, dropout: float, lr: float) -> tf.keras.Model:
    model = Sequential([
        SimpleRNN(units1, return_sequences=True, input_shape=input_shape),
        Dropout(dropout),
        SimpleRNN(units2, return_sequences=False),
        Dropout(dropout),
        Dense(OUTPUT_HOURS)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(input_shape: Tuple[int,int], units1: int, units2: int, dropout: float, lr: float) -> tf.keras.Model:
    model = Sequential([
        LSTM(units1, return_sequences=True, input_shape=input_shape),
        Dropout(dropout),
        LSTM(units2, return_sequences=False),
        Dropout(dropout),
        Dense(OUTPUT_HOURS)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

def random_search_rnn_lstm(model_type: str,
                           X_train, y_train, X_val, y_val,
                           n_trials: int = 40):
    """
    Búsqueda aleatoria simple para RNN/LSTM:
    - muestrea combinaciones de hiperparámetros
    - entrena con early stopping y evalúa en validation
    - guarda el mejor modelo (según val loss)
    """
    log(f"\nIniciando búsqueda aleatoria para {model_type.upper()} ({n_trials} trials)...")
    input_shape = (X_train.shape[1], X_train.shape[2])

    # espacios de búsqueda (ajustables)
    units1_choices = [32, 64, 96, 128, 256]
    units2_choices = [32, 64, 96, 128]
    dropout_choices = [0.1, 0.15, 0.2, 0.25, 0.3]
    lr_choices = [1e-3, 5e-4, 2e-4, 1e-4]
    batch_choices = [16, 32, 64]
    epochs_max = 100

    best_val_loss = np.inf
    best_model = None
    best_params = None
    history_records = []

    for trial in range(1, n_trials + 1):
        # muestreo aleatorio
        units1 = random.choice(units1_choices)
        units2 = random.choice(units2_choices)
        dropout = random.choice(dropout_choices)
        lr = random.choice(lr_choices)
        batch_size = random.choice(batch_choices)

        params = {'units1': units1, 'units2': units2, 'dropout': dropout, 'lr': lr, 'batch_size': batch_size}
        log(f" Trial {trial}/{n_trials} - params: {params}")

        # construir
        if model_type == 'rnn':
            model = build_rnn_model(input_shape, units1, units2, dropout, lr)
        else:
            model = build_lstm_model(input_shape, units1, units2, dropout, lr)

        # callbacks
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

        # entrenar
        try:
            hist = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=epochs_max,
                batch_size=batch_size,
                callbacks=[early_stop, reduce_lr],
                verbose=0
            )
            val_loss = min(hist.history['val_loss'])
            log(f"  -> val_loss: {val_loss:.6f} (mejor)")

            history_records.append({'trial': trial, 'params': params, 'val_loss': float(val_loss), 'history': hist.history})
            # actualizar mejor
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model = model
                best_params = params
                # guardar modelo intermedio
                tmp_path = OUTPUT_DIR / f"{model_type}_best_intermediate.h5"
                model.save(tmp_path)
                log(f"  Nuevo best -> val_loss {best_val_loss:.6f} guardado en {tmp_path}")
        except Exception as e:
            log(f"  Error en trial {trial}: {e}")
            traceback.print_exc()
            continue

    # guardar historial de búsqueda
    pd.DataFrame([{'trial': r['trial'], **r['params'], 'val_loss': r['val_loss']} for r in history_records]).to_csv(OUTPUT_DIR / f'{model_type}_random_search_history.csv', index=False)
    with open(OUTPUT_DIR / f'{model_type}_random_search_meta.json', 'w') as fh:
        json.dump({'best_params': best_params, 'best_val_loss': float(best_val_loss)}, fh, indent=2)

    # devolver mejor modelo y meta
    return best_model, best_params, history_records

# ---------------------- MAIN: FLUJO COMPLETO ----------------------
def main():
    log("=== Inicio entrenamiento y búsqueda hiperparámetros ===")
    log(f"Modo: {MODE} - Estación: {STATION_NAME if MODE=='station' else 'combined'}")

    # 1) Cargar RF/XGB
    X_rf, y_rf = load_rf_xgb_data(MODE, STATION_NAME if MODE == 'station' else None)
    log(f"RF/XGB cargado: X={X_rf.shape}, y={y_rf.shape}")

    # 2) Split temporal
    (X_rf_train, y_rf_train), (X_rf_val, y_rf_val), (X_rf_test, y_rf_test) = temporal_train_val_test_split(X_rf, y_rf, VAL_SIZE, TEST_SIZE)
    log(f"Split RF/XGB - Train: {len(X_rf_train)} Val: {len(X_rf_val)} Test: {len(X_rf_test)}")

    # 3) Escalado si aplica (se recomienda para XGBoost en algunos casos; para RF no es necesario)
    scaler_rf = None
    if True:
        scaler_rf = StandardScaler()
        X_rf_train_scaled = pd.DataFrame(scaler_rf.fit_transform(X_rf_train), index=X_rf_train.index, columns=X_rf_train.columns)
        X_rf_val_scaled = pd.DataFrame(scaler_rf.transform(X_rf_val), index=X_rf_val.index, columns=X_rf_val.columns)
        X_rf_test_scaled = pd.DataFrame(scaler_rf.transform(X_rf_test), index=X_rf_test.index, columns=X_rf_test.columns)
        joblib.dump(scaler_rf, OUTPUT_DIR / 'scaler_rf_xgb.pkl')
        X_rf_train_use, X_rf_val_use, X_rf_test_use = X_rf_train_scaled, X_rf_val_scaled, X_rf_test_scaled
        log("Scaler RF/XGB guardado.")
    else:
        X_rf_train_use, X_rf_val_use, X_rf_test_use = X_rf_train, X_rf_val, X_rf_test

    # ---------------------------------------------------------
    # Randomized search RF (usa X_rf_train_use & y_rf_train)
    try:
        rf_best, rf_search = randomized_search_rf(X_rf_train_use, y_rf_train)
    except Exception as e:
        log(f"Error en RandomizedSearch RF: {e}")
        traceback.print_exc()
        rf_best = None

    # Evaluar RF best en test (si existe)
    if rf_best is not None:
        y_pred_rf_test = rf_best.predict(X_rf_test_use)
        rf_metrics = evaluate_model(y_rf_test.values, y_pred_rf_test, "RF", "test")
        joblib.dump(rf_best, OUTPUT_DIR / 'rf_model_final.pkl')
    else:
        rf_metrics = {'MAE': None, 'RMSE': None, 'R2': None}

    # ---------------------------------------------------------
    # Randomized search XGBoost (envolviendo con MultiOutputRegressor)
    try:
        xgb_best_wrapper, xgb_search = randomized_search_xgb(X_rf_train_use, y_rf_train)
    except Exception as e:
        log(f"Error en RandomizedSearch XGB: {e}")
        traceback.print_exc()
        xgb_best_wrapper = None

    if xgb_best_wrapper is not None:
        y_pred_xgb_test = xgb_best_wrapper.predict(X_rf_test_use)
        xgb_metrics = evaluate_model(y_rf_test.values, y_pred_xgb_test, "XGB", "test")
        joblib.dump(xgb_best_wrapper, OUTPUT_DIR / 'xgb_model_final.pkl')
    else:
        xgb_metrics = {'MAE': None, 'RMSE': None, 'R2': None}

    # ---------------------------------------------------------
    # Cargar datos RNN/LSTM (.npz)
    X_rnn, y_rnn = load_rnn_lstm_data(MODE, STATION_NAME if MODE == 'station' else None)
    log(f"RNN/LSTM cargado: X={X_rnn.shape}, y={y_rnn.shape}")
    (X_rnn_train, y_rnn_train), (X_rnn_val, y_rnn_val), (X_rnn_test, y_rnn_test) = temporal_train_val_test_split(X_rnn, y_rnn, VAL_SIZE, TEST_SIZE)
    log(f"Split RNN - Train: {len(X_rnn_train)} Val: {len(X_rnn_val)} Test: {len(X_rnn_test)}")

    # Escalado RNN (por feature)
    scaler_rnn = None
    try:
        X_rnn_train_s, X_rnn_val_s, X_rnn_test_s, scaler_rnn = scale_features(X_rnn_train, X_rnn_val, X_rnn_test)
        joblib.dump(scaler_rnn, OUTPUT_DIR / 'scaler_rnn_lstm.pkl')
        log("Scaler RNN guardado.")
    except Exception as e:
        log(f"Error escalando RNN data: {e}")
        traceback.print_exc()
        X_rnn_train_s, X_rnn_val_s, X_rnn_test_s = X_rnn_train, X_rnn_val, X_rnn_test

    # ---------------------------------------------------------
    # Random search RNN
    try:
        rnn_best_model, rnn_best_params, rnn_history = random_search_rnn_lstm('rnn', X_rnn_train_s, y_rnn_train, X_rnn_val_s, y_rnn_val, n_trials=N_TRIALS_RNN)
        if rnn_best_model is not None:
            y_pred_rnn_test = rnn_best_model.predict(X_rnn_test_s)
            rnn_metrics = evaluate_model(y_rnn_test, y_pred_rnn_test, "RNN", "test")
            rnn_best_model.save(OUTPUT_DIR / 'rnn_model_final.h5')
        else:
            rnn_metrics = {'MAE': None, 'RMSE': None, 'R2': None}
    except Exception as e:
        log(f"Error en búsqueda RNN: {e}")
        traceback.print_exc()
        rnn_metrics = {'MAE': None, 'RMSE': None, 'R2': None}

    # Random search LSTM
    try:
        lstm_best_model, lstm_best_params, lstm_history = random_search_rnn_lstm('lstm', X_rnn_train_s, y_rnn_train, X_rnn_val_s, y_rnn_val, n_trials=N_TRIALS_LSTM)
        if lstm_best_model is not None:
            y_pred_lstm_test = lstm_best_model.predict(X_rnn_test_s)
            lstm_metrics = evaluate_model(y_rnn_test, y_pred_lstm_test, "LSTM", "test")
            lstm_best_model.save(OUTPUT_DIR / 'lstm_model_final.h5')
        else:
            lstm_metrics = {'MAE': None, 'RMSE': None, 'R2': None}
    except Exception as e:
        log(f"Error en búsqueda LSTM: {e}")
        traceback.print_exc()
        lstm_metrics = {'MAE': None, 'RMSE': None, 'R2': None}

    # ---------------------------------------------------------
    # Guardar comparativa final
    comparativa = pd.DataFrame({
        'RF': rf_metrics,
        'XGB': xgb_metrics,
        'RNN': rnn_metrics,
        'LSTM': lstm_metrics
    }).T
    comparativa.to_csv(OUTPUT_DIR / 'comparativa_metricas_test.csv')
    log("\n=== BÚSQUEDA COMPLETA ===")
    log(comparativa.to_string())

    # Guardar meta experimentos
    meta = {
        'rf_best_params_file': str(OUTPUT_DIR / 'rf_best_params.json'),
        'xgb_best_params_file': str(OUTPUT_DIR / 'xgb_best_params.json'),
        'rnn_best_params': rnn_best_params if 'rnn_best_params' in locals() else None,
        'lstm_best_params': lstm_best_params if 'lstm_best_params' in locals() else None,
        'comparativa': comparativa.to_dict()
    }
    with open(OUTPUT_DIR / 'hypersearch_summary.json', 'w') as fh:
        json.dump(meta, fh, indent=2)

    log(f"\nResultados, modelos y logs guardados en: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()

Python(75637) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
2026-03-02 18:57:38.670199: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


=== Inicio entrenamiento y búsqueda hiperparámetros ===
Modo: station - Estación: T1_E1_Alicante
RF/XGB cargado: X=(140113, 1656), y=(140113, 72)
Split RF/XGB - Train: 101231 Val: 17865 Test: 21017
Scaler RF/XGB guardado.

Iniciando RandomizedSearch para RandomForest (esto puede tardar)...
Fitting 4 folds for each of 60 candidates, totalling 240 fits


Python(75646) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(75648) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(75649) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(75650) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(75651) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(75652) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time= 2.4min
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time= 5.4min
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time= 8.8min
[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=400; total time= 9.3min
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=12.6min
[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=400; total time=20.3min
[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=400; total time=31.8min
[CV] END bootstrap=True, max_depth=3